In [1]:
from WeatherDMD.evaluate_wb2 import evaluate_wb2
from pyprojroot import here
import os
import pytest
import xarray as xr
from WeatherDMD.constants import (
    wb2_variables,
    wb2_forecast_dimensions,
    wb2_obs_dimensions
)

input_data_path  = os.path.join(here(), "tests/evaluate_wb2/data/input")
output_data_path = os.path.join(here(), "tests/evaluate_wb2/data/output")

In [ ]:
# """
# Temporarily save NetCDF files as Zarr files for testing.
# """
# obs_path      = os.path.join(input_data_path, "era5_slice_test.nc")
# forecast_path = os.path.join(input_data_path, "era5_dmd_forecast_test.nc")

# obs      = xr.open_dataset(obs_path)
# forecast = xr.open_dataset(forecast_path)

# obs_path      = os.path.join(input_data_path, "era5_slice_test.zarr")
# forecast_path = os.path.join(input_data_path, "era5_dmd_forecast_test.zarr")

# obs.to_zarr(obs_path, mode="w")
# forecast.to_zarr(forecast_path, mode="w")

In [2]:
def test_format_of_files(temp_data):
    """
    Test the format of the incoming data: names of dimensions, structure, etc
    """

    obs      = xr.open_dataset(os.path.join(input_data_path, "era5_slice_test.zarr"))
    forecast = xr.open_dataset(os.path.join(input_data_path, "era5_dmd_forecast_test.zarr"))

    # Check that obs and forecast variables are members of a list of accepted variables
    assert set(obs.data_vars).issubset(wb2_forecast_dimensions), "Observed variables are not all accepted"
    assert set(forecast.data_vars).issubset(wb2_forecast_dimensions), "Forecasted variables are not all accepted"    

    # Check that the dimensions of the forecast and obs files are correct
    assert set(obs.dims).issubset(wb2_obs_dimensions), "Observed dimensions are not all accepted"
    assert set(forecast.dims).issubset(wb2_forecast_dimensions), "Forecasted dimensions are not all accepted"    

    # Check that data type of the "time" dimension is datetime64[ns]
    assert obs.time.dtype == "datetime64[ns]", "Time dimension in observations is not datetime64[ns]"
    assert forecast.time.dtype == "datetime64[ns]", "Time dimension in forecast is not datetime64[ns]"        

    assert obs.latitude.dtype == "float32", "Latitude dimension in observations is not float32"
    assert forecast.latitude.dtype == "float32", "Latitude dimension in forecast is not float32"

    assert obs.longitude.dtype == "float32", "Latitude dimension in observations is not float32"
    assert forecast.longitude.dtype == "float32", "Latitude dimension in forecast is not float32"     

    assert obs.level.dtype == "int64", "level dimension in observations is not int64"
    assert forecast.level.dtype == "int64", "level dimension in forecast is not int64"

    # Test that each data variable in observations is of type float32
    for var in obs.data_vars:
        assert obs[var].dtype == "float32", f"Variable {var} in observations is not float32"

    # Test that each data variable in forecast is of type float32
    for var in forecast.data_vars:
        assert forecast[var].dtype == "float32", f"Variable {var} in forecast is not float32"        



def test_evaluate_wb2(temp_data):
    """
    Test the evaluate_wb2 function.
    """
    obs_path = os.path.join(input_data_path, "era5_slice_test.zarr")
    forecast_path = os.path.join(input_data_path, "era5_dmd_forecast_test.zarr")

    evaluate_wb2(obs_path, forecast_path, output_dir=output_data_path)


In [9]:
obs = xr.open_dataset(os.path.join(input_data_path, "era5_slice_test.zarr"))

obs

# Check that data type of the "time" dimension is datetime64[ns]
# assert obs.time.dtype == "datetime64[ns]", "Time dimension is not datetime64[ns]"

obs.latitude.dtype == "float32"

/opt/homebrew/anaconda3/envs/dmd/lib/python3.12/site-packages/xarray/backends/plugins.py:159: RuntimeWarning: 'netcdf4' fails while guessing
  warnings.warn(f"{engine!r} fails while guessing", RuntimeWarning)
/opt/homebrew/anaconda3/envs/dmd/lib/python3.12/site-packages/xarray/backends/plugins.py:159: RuntimeWarning: 'scipy' fails while guessing
  warnings.warn(f"{engine!r} fails while guessing", RuntimeWarning)


True

In [10]:
obs

<xarray.Dataset>
Dimensions:       (time: 50, level: 1, latitude: 72, longitude: 144)
Coordinates:
  * latitude      (latitude) float32 89.0 86.5 84.0 81.5 ... -83.5 -86.0 -88.5
  * level         (level) int64 1000
  * longitude     (longitude) float32 1.0 3.5 6.0 8.5 ... 353.5 356.0 358.5
  * time          (time) datetime64[ns] 2020-01-01 ... 2020-01-13T06:00:00
Data variables:
    geopotential  (time, level, latitude, longitude) float32 ...
    temperature   (time, level, latitude, longitude) float32 ...

In [20]:
# Check that obs variables are members of a list of accepted variables
accepted_variables = ["temperature","geopotential","specific_humidity","u_component_of_wind","v_component_of_wind"]

assert set(obs.data_vars).issubset(accepted_variables), "Observed variables are not all accepted"

# Check that dimensions are members of a list of accepted dimensions
accepted_dimensions = ["time","latitude","longitude","level"]

assert set(obs.dims).issubset(accepted_dimensions), "Observed dimensions are not all accepted"

In [ ]:
forecast = xr.open_dataset(os.path.join(input_data_path, "era5_dmd_forecast_test.zarr"))

forecast